In [ ]:
import os
print(os.getcwd())
print('-'*80)
# !pip install --force-reinstall ./data_pipeline-0.0.1-py3-none-any.whl

In [ ]:
import pyspark
print(f'pyspark.__version__ = {pyspark.__version__}')
import importlib_metadata
delta_version = importlib_metadata.version("delta_spark")
print(f'delta_version = {delta_version}')

In [ ]:
from pyspark.sql import SparkSession
from pyspark import SparkContext

# Configure spark session with Delta Lake JARs
spark = SparkSession.builder \
    .appName("GenericSparkApp") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.1,org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/delta") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localstack:4566") \
    .config("spark.hadoop.fs.s3a.access.key", "test") \
    .config("spark.hadoop.fs.s3a.secret.key", "test") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true") \
    .getOrCreate()

print("spark session created with Delta Lake support!")

In [ ]:
def verify_parquet_optimizations(spark):
    """
    Verify that Parquet optimizations are enabled and working.
    
    Args:
        spark: SparkSession object
    """
    print("=" * 80)
    print("Parquet Optimizations Verification")
    print("=" * 80)
    
    # Check Parquet configurations
    print("\n📊 Parquet Configuration Status:")
    print(f"  ✓ Vectorized Reader: {spark.conf.get('spark.sql.parquet.enableVectorizedReader', 'Not set')}")
    print(f"  ✓ Filter Pushdown: {spark.conf.get('spark.sql.parquet.filterPushdown', 'Not set')}")
    print(f"  ✓ Column Index Reader: {spark.conf.get('spark.sql.parquet.columnIndexReader.enabled', 'Not set')}")
    print(f"  ✓ Page Index: {spark.conf.get('spark.sql.parquet.enablePageIndex', 'Not set')}")
    print(f"  ✓ Record Filter: {spark.conf.get('spark.sql.parquet.recordFilter.enabled', 'Not set')}")
    print(f"  ✓ Columnar Reader Batch Size: {spark.conf.get('spark.sql.parquet.columnarReaderBatchSize', 'Not set')}")
    
    # Check Databricks IO Cache (may not be available)
    print("\n💾 Databricks IO Cache Status (may not work in standard Spark):")
    io_cache_enabled = spark.conf.get('spark.databricks.io.cache.enabled', 'Not set')
    print(f"  {'✓' if io_cache_enabled == 'true' else '✗'} IO Cache Enabled: {io_cache_enabled}")
    if io_cache_enabled == 'true':
        print(f"  ✓ Max Disk Usage: {spark.conf.get('spark.databricks.io.cache.maxDiskUsage', 'Not set')}")
        print(f"  ✓ Max Metadata Cache: {spark.conf.get('spark.databricks.io.cache.maxMetaDataCache', 'Not set')}")
    
    print("\n" + "=" * 80)
    print("Note: Parquet optimizations are working even if 'Parquet IO Cache'")
    print("doesn't appear in Storage tab (that's a Databricks-specific UI feature).")
    print("Check Spark UI SQL tab for query execution plans showing vectorized operations.")
    print("=" * 80)

# Verify optimizations are active
verify_parquet_optimizations(spark)
